# Cobertura_Declaracao — declaração de cobertura da extração (insumo da Gold)

Tarefa do Job `v-energy-grid-360`, entre `Bronze_to_Silver` e `Silver_to_gold`. Gera a declaração de cobertura `gold-cobertura-v2` do lote Silver ativo e a entrega à Gold por *task values*, em vez de um operador preencher o JSON à mão.

**O que confere, para consumo (`leituras`) e faturamento (`faturas`):**
1. A reconciliação da Bronze do lote ativo fecha (`delta` zero e gravadas + rejeitadas = linhas de dados).
2. A Silver fecha com a Bronze (gravadas = aceitas + quarentena da Silver).
3. A Gold conta todos os registros aceitos pela Silver na janela (nenhum excluído por UC nula/de teste ou sem competência).
4. Todos os meses da janela têm registros em cada domínio.
5. Identidade da declaração: origem = aceitos + excluídos + quarentena.

Se qualquer conferência falhar, a tarefa **falha** e o Job para antes da Gold: não se declara cobertura que os números não sustentam. `registros_excluidos` é só o que a origem deixou de entregar; rejeitados por qualidade (Bronze e quarentena da Silver) entram em `registros_quarentena`. A data de corte é a data da carga na Bronze; o hash é o SHA-256 dos hashes dos arquivos de origem. Nenhum valor de negócio é impresso.

**Saídas (task values):** `declaracao_json`, `cobertura_inicio`, `cobertura_fim`, `pv_inicio_p0`, `pv_fim_p0`, `pv_inicio_p1`, `pv_fim_p1`. A janela vem das competências observadas; as janelas de preço × volume são os 12 primeiros e os 12 últimos meses, e ficam vazias se a janela tiver menos de 24 meses.

In [ ]:
import json, hashlib
from datetime import timezone
from pyspark.sql import functions as F

CAT = "workspace"

def chk(nome, ok):
    print(("PASS " if ok else "FAIL ") + nome)
    return bool(ok)

pub = spark.table(f"{CAT}.silver.controle_publicacao").filter(F.col("chave") == "silver").collect()[0].asDict()
run_silver, lote = pub["run_ativo"], pub["lote_ativo"]
print("Silver:", run_silver, "| lote Bronze:", lote)

rec_all = spark.table(f"{CAT}.bronze_hist.reconciliacao")
col_lote = "_lote_id" if rec_all.filter(F.col("_lote_id") == lote).limit(1).count() else "_lote_origem"
rec_lote = rec_all.filter(F.col(col_lote) == lote)
quar = spark.table(f"{CAT}.silver_hist.quarentena").filter(F.col("_run_id") == run_silver)

DOMINIOS = [("consumo", "leituras"), ("faturamento", "faturas")]
uc_valida = lambda df: df.filter(F.col("id_uc").isNotNull() & ~F.col("id_uc").startswith("TESTE"))

# Janela: competencias observadas nos dois dominios
limites = []
for _, tabela in DOMINIOS:
    d = uc_valida(spark.table(f"{CAT}.silver_publicada.{tabela}")).filter(F.col("mes_referencia").isNotNull())
    limites.append(d.agg(F.min("mes_referencia"), F.max("mes_referencia")).first())
INICIO = min(x[0] for x in limites)
FIM = max(x[1] for x in limites)

def indice(c): return int(c[:4]) * 12 + int(c[5:7]) - 1
def competencia(i): return f"{i // 12:04d}-{i % 12 + 1:02d}"
n_meses = indice(FIM) - indice(INICIO) + 1
print("janela de cobertura:", INICIO, "a", FIM, f"({n_meses} meses)")

declaracoes, tudo_ok = [], True
for dominio, tabela in DOMINIOS:
    print(f"--- {dominio} ({tabela})")
    r = rec_lote.filter(F.lower(F.col("_tabela")) == tabela)
    if not chk("reconciliação Bronze encontrada para o lote", r.limit(1).count() == 1):
        tudo_ok = False
        continue
    agg = r.agg(F.sum("linhas_dados").alias("origem"), F.sum("gravadas").alias("gravadas"),
                F.sum("rejeitadas").alias("rejeitadas"), F.sum("delta").alias("delta"),
                F.max("_data_carga").alias("carga")).first()
    origem, gravadas, rej_bronze = int(agg["origem"]), int(agg["gravadas"]), int(agg["rejeitadas"] or 0)
    silver = spark.table(f"{CAT}.silver_publicada.{tabela}")
    silver_total = silver.count()
    q_silver = quar.filter(F.col("_tabela_origem") == tabela).count()
    no_gold = (uc_valida(silver).filter(F.col("mes_referencia").between(INICIO, FIM)))
    aceitos = no_gold.count()
    meses_com_registro = no_gold.select("mes_referencia").distinct().count()
    excluidos = silver_total - aceitos
    quarentena = rej_bronze + q_silver
    corte = agg["carga"].replace(tzinfo=timezone.utc)
    ok = chk("Bronze: delta zero em todos os arquivos", int(agg["delta"] or 0) == 0)
    ok &= chk("Bronze: gravadas + rejeitadas = linhas de dados", gravadas + rej_bronze == origem)
    ok &= chk("Silver: gravadas na Bronze = aceitos + quarentena da Silver", gravadas == silver_total + q_silver)
    ok &= chk("Gold conta todos os registros aceitos pela Silver (excluídos = 0)", excluidos == 0)
    ok &= chk("todos os meses da janela têm registros", meses_com_registro == n_meses)
    ok &= chk("identidade: origem = aceitos + excluídos + quarentena", origem == aceitos + excluidos + quarentena)
    ok &= chk("corte posterior ao início do mês seguinte à janela", corte.isoformat() >= f"{competencia(indice(FIM) + 1)}-01T00:00:00+00:00")
    arquivos = sorted(f"{x['_arquivo_origem']}|{x['_aba_origem'] or ''}|{x['hash_sha256']}" for x in r.collect())
    inventario = hashlib.sha256("\n".join(arquivos).encode("utf-8")).hexdigest()
    tudo_ok &= ok
    declaracoes.append({
        "dominio": dominio, "recorte": "DOMINIO", "inicio": INICIO, "fim": FIM, "estado": "FECHADO",
        "data_corte": corte.isoformat(), "referencia_evidencia": f"bronze_hist.reconciliacao/{lote}/{tabela}",
        "natureza_certificacao": "EXTRACAO",
        "reconciliacao_extracao": {"registros_origem": origem, "registros_aceitos": aceitos,
            "registros_excluidos": excluidos, "registros_quarentena": quarentena, "hash_inventario": inventario}})

if not tudo_ok or len(declaracoes) != len(DOMINIOS):
    raise RuntimeError("Cobertura não declarada: as conferências acima não fecham. Corrija a origem antes da Gold.")

pacote = {"schema_version": "gold-cobertura-v2", "run_id_silver": run_silver, "lote_bronze_silver": lote, "declaracoes": declaracoes}
texto = json.dumps(pacote, ensure_ascii=False, separators=(",", ":"))
assert len(texto) < 40000, "declaração excede o limite de task value"

# Janelas de preço x volume: 12 primeiros e 12 ultimos meses (vazias se houver menos de 24 meses)
if n_meses >= 24:
    pv = (INICIO, competencia(indice(INICIO) + 11), competencia(indice(FIM) - 11), FIM)
else:
    pv = ("", "", "", "")
valores = {"declaracao_json": texto, "cobertura_inicio": INICIO, "cobertura_fim": FIM,
           "pv_inicio_p0": pv[0], "pv_fim_p0": pv[1], "pv_inicio_p1": pv[2], "pv_fim_p1": pv[3]}
for chave, valor in valores.items():
    dbutils.jobs.taskValues.set(key=chave, value=valor)
print("PASS declaração gerada e entregue à Gold (tamanho em caracteres):", len(texto))
print("janelas de preço x volume:", pv)